# LearnRAG: a simple local RAG pipeline

This notebook retrieves relevant document passages and gives them to a local language model to generate an answer. It uses public models with no paid API key. Initial model downloads require Internet access and several GB of space. A Colab GPU is recommended.

This notebook runs MiniLM retrieval and Qwen generation on the included sample document. Saved outputs are observations, not accuracy scores. Component tests use test doubles and do not measure model quality; see `results/` for separate captured runs and review notes.


## 1. Get the code and install dependencies

In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. The next cell clones LearnRAG only if the current folder does not already contain its script. Existing files are reused; it does not pull or overwrite changes.


In [1]:
# Setup only: reuse an existing checkout rather than overwriting local edits.
from pathlib import Path
import os, subprocess

if not Path("simple_rag.py").is_file():
    project = Path.cwd() / "LearnRAG"
    if not project.exists():
        subprocess.run(["git", "clone", "https://github.com/nafismam/LearnRAG.git", str(project)], check=True)
    if not (project / "simple_rag.py").is_file():
        raise RuntimeError("LearnRAG exists but does not contain simple_rag.py; select the correct project directory.")
    os.chdir(project)
print("Project directory:", Path.cwd())


Project directory: /workspace/LearnRAG


In [2]:
# Install the packages used by the implementation below. Initial model downloads happen later.
%pip install -r requirements.txt



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Inspect the data and chunks
Chunks retain a relative file path and a unique ID. Generic character windows are used here, rather than statutory section extraction.


In [3]:
# Import the real pipeline code from simple_rag.py; this notebook does not duplicate it.
# Preview loading/chunking before spending time downloading generation models.
from simple_rag import RAGConfig, SimpleRAG, load_documents, chunk_documents
config = RAGConfig(temperature=0.0)
docs = load_documents(config.data_dir)
chunks = chunk_documents(docs, config.chunk_size, config.chunk_overlap)
print("Chunks:", len(chunks))
print(chunks[0])


[load] 1 document(s) loaded from 'data'
[chunk] 3 chunk(s) created
Chunks: 3
Chunk(text="# Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation (RAG) retrieves passages from a document collection, adds them to a language model prompt, and generates an answer using that context. It can provide information outside a model's training data, but does not guarantee accurate answers.\n\n# This demonstration\n\nThis pipeline uses sentence-transformers/all-MiniLM-L6-v2 as its embedding model. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vec", source='rag_notes.md', chunk_id=0)


## 3. Build the index and load the generator
MiniLM embeds chunks into normalized vectors stored in NumPy. Qwen generates answers from retrieved context. The first run downloads several GB; a Colab GPU is recommended. CPU generation can be slow.


In [4]:
# This performs indexing and loads the generator once; later ask() calls reuse them.
rag = SimpleRAG(config)


[load] 1 document(s) loaded from 'data'
[chunk] 3 chunk(s) created


/workspace/learnrag-runtime/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[embed] loading embedding model 'sentence-transformers/all-MiniLM-L6-v2' ...


[index] vector matrix shape: (3, 384)
[llm] loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' (first run downloads it) ...


`torch_dtype` is deprecated! Use `dtype` instead!


## 4. Retrieve, augment and generate
Inspect the actual source text alongside each answer. Numbered citations are model output, not automatically verified evidence.


In [5]:
# RETRIEVE -> AUGMENT -> GENERATE. Inspect source text separately from generated claims.
result = rag.ask("Which embedding model does this pipeline use?")
print(result["sources"])


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Q: Which embedding model does this pipeline use?
----------------------------------------------------------------------
Retrieved chunks:
  [1] score=0.490  rag_notes.md: # Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) retrieves passages from a doc...
  [2] score=0.305  rag_notes.md: odel. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vec...
  [3] score=0.129  rag_notes.md: monstration, not all RAG systems.  # Limitations  Retrieved passages may be irrelevant or incomplete...
----------------------------------------------------------------------
A: The pipeline uses sentence-transformers/all-MiniLM-L6-v2 as its embedding model.
[{'source': 'rag_notes.md', 'score': 0.49042820930480957, 'text': "# Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation (RAG) retrieves passages from a document collection, adds them to a language model prompt, and generates an answer using that context. It can provide information

In [6]:
# This checks observed explanatory behavior, not automatically graded faithfulness.
rag.ask("What is Retrieval-Augmented Generation and why is it useful?")



Q: What is Retrieval-Augmented Generation and why is it useful?
----------------------------------------------------------------------
Retrieved chunks:
  [1] score=0.577  rag_notes.md: # Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) retrieves passages from a doc...
  [2] score=0.295  rag_notes.md: odel. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vec...
  [3] score=0.278  rag_notes.md: monstration, not all RAG systems.  # Limitations  Retrieved passages may be irrelevant or incomplete...
----------------------------------------------------------------------
A: Retrieval-Augmented Generation (RAG) is a technique that combines machine learning models with external knowledge bases to generate human-like responses. It works by retrieving relevant information from a large corpus of documents and then using this context to inform the generation process. This approach allows RAG to provide more comprehensive and accurate 

{'question': 'What is Retrieval-Augmented Generation and why is it useful?',
 'answer': 'Retrieval-Augmented Generation (RAG) is a technique that combines machine learning models with external knowledge bases to generate human-like responses. It works by retrieving relevant information from a large corpus of documents and then using this context to inform the generation process. This approach allows RAG to provide more comprehensive and accurate answers than traditional models alone, especially when dealing with complex queries or when there is insufficient training data available. However, it also has limitations such as potential inaccuracies due to irrelevant or incomplete retrieved passages and issues with model adherence to instructions or citation requirements.',
 'sources': [{'source': 'rag_notes.md',
   'score': 0.5772419571876526,
   'text': "# Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation (RAG) retrieves passages from a document collection, adds them to a l

## 5. Check an unsupported question
The prompt asks the model to abstain when context lacks an answer. This is an observation to inspect, not a guaranteed passing test. Retrieval always returns top-k chunks and currently has no confidence gate.


In [7]:
# Negative example: the source does not answer this question. Abstention is requested, not enforced.
rag.ask("Who won the 1998 FIFA World Cup?")



Q: Who won the 1998 FIFA World Cup?
----------------------------------------------------------------------
Retrieved chunks:
  [1] score=0.020  rag_notes.md: monstration, not all RAG systems.  # Limitations  Retrieved passages may be irrelevant or incomplete...
  [2] score=0.009  rag_notes.md: # Retrieval-Augmented Generation  Retrieval-Augmented Generation (RAG) retrieves passages from a doc...
  [3] score=-0.005  rag_notes.md: odel. It converts text chunks into normalized 384-dimensional vectors. A NumPy matrix stores the vec...
----------------------------------------------------------------------
A: Based on the given context, there is no specific mention of who won the 1998 FIFA World Cup. Therefore, I don't know based on the provided documents.


{'question': 'Who won the 1998 FIFA World Cup?',
 'answer': "Based on the given context, there is no specific mention of who won the 1998 FIFA World Cup. Therefore, I don't know based on the provided documents.",
 'sources': [{'source': 'rag_notes.md',
   'score': 0.020306911319494247,
   'text': 'monstration, not all RAG systems.\n\n# Limitations\n\nRetrieved passages may be irrelevant or incomplete. A model may ignore instructions or invent citations. Inspect both the retrieved passages and the generated answer. This example does not automatically validate claims or citations and is not intended for legal advice.',
   'chunk_id': 2},
  {'source': 'rag_notes.md',
   'score': 0.008632994256913662,
   'text': "# Retrieval-Augmented Generation\n\nRetrieval-Augmented Generation (RAG) retrieves passages from a document collection, adds them to a language model prompt, and generates an answer using that context. It can provide information outside a model's training data, but does not guaran

## Reading the observed answers

Inspect whether each claim is supported by the retrieved text. An answer can be plausible while adding unsupported claims, omitting requested citations, or failing to abstain. The small sample corpus has only three chunks, and top-k=3 returns all of them; this exercise does not measure selection quality on a larger collection. CPU/GPU differences and model-library versions can change output.
